In [1]:
# mcp_orchestrator.py

import requests
import re

# Both agents (but we treat them as TOOL servers, not agents)
SERVERS = [
    "http://localhost:8000",  # math-agent
    "http://localhost:8001"   # finance-agent
]


# -------------------------------
# STEP 1: Discover ALL tools (MCP)
# -------------------------------
def discover_tools():
    all_tools = []

    for server in SERVERS:
        res = requests.get(f"{server}/tools").json()

        for tool in res["tools"]:
            tool["server"] = server   # attach origin
            all_tools.append(tool)

    return all_tools


# -------------------------------
# STEP 2: Naive Tool Matching
# -------------------------------
def find_tool(query, tools):
    for tool in tools:
        if tool["name"] in query:
            return tool

    # fallback guess
    if "interest" in query:
        return next(t for t in tools if "interest" in t["name"])

    if "add" in query:
        return next(t for t in tools if "add" in t["name"])

    return None


# -------------------------------
# STEP 3: Extract numbers (weak)
# -------------------------------
def extract_numbers(query):
    return list(map(float, re.findall(r"\d+\.?\d*", query)))



In [2]:

# -------------------------------
# STEP 4: Execute SINGLE tool
# -------------------------------
def execute(query):
    tools = discover_tools()

    print("\n🔍 DISCOVERED TOOLS:", tools)

    tool = find_tool(query, tools)

    if not tool:
        return "❌ No tool found"

    numbers = extract_numbers(query)

    print("🧠 SELECTED TOOL:", tool["name"])
    print("🔢 EXTRACTED NUMBERS:", numbers)

    # ⚠️ BIG LIMITATION: assumes ONE tool only
    if tool["name"] == "calculate_interest":
        payload = {
            "tool": "calculate_interest",
            "args": {
                "principal": numbers[0],
                "rate": numbers[1],
                "time": numbers[2]
            }
        }

    elif tool["name"] == "add":
        payload = {
            "tool": "add",
            "args": {
                "a": numbers[0],
                "b": numbers[1]
            }
        }

    else:
        return "❌ Unsupported tool"

    response = requests.post(f"{tool['server']}/invoke", json=payload).json()

    return response

In [3]:
query = "calculate interest for 1000 at 5 for 2 years"
print(execute(query))


🔍 DISCOVERED TOOLS: [{'name': 'add', 'description': 'Add two numbers', 'input_schema': {'type': 'object', 'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}}, 'required': ['a', 'b']}, 'server': 'http://localhost:8000'}, {'name': 'calculate_interest', 'description': 'Simple interest', 'input_schema': {'type': 'object', 'properties': {'principal': {'type': 'number'}, 'rate': {'type': 'number'}, 'time': {'type': 'number'}}, 'required': ['principal', 'rate', 'time']}, 'server': 'http://localhost:8001'}]
🧠 SELECTED TOOL: calculate_interest
🔢 EXTRACTED NUMBERS: [1000.0, 5.0, 2.0]
{'result': 100.0}


In [4]:
query = "calculate interest for 1000 at 5 for 2 years and then add 100"
print(execute(query))


🔍 DISCOVERED TOOLS: [{'name': 'add', 'description': 'Add two numbers', 'input_schema': {'type': 'object', 'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}}, 'required': ['a', 'b']}, 'server': 'http://localhost:8000'}, {'name': 'calculate_interest', 'description': 'Simple interest', 'input_schema': {'type': 'object', 'properties': {'principal': {'type': 'number'}, 'rate': {'type': 'number'}, 'time': {'type': 'number'}}, 'required': ['principal', 'rate', 'time']}, 'server': 'http://localhost:8001'}]
🧠 SELECTED TOOL: add
🔢 EXTRACTED NUMBERS: [1000.0, 5.0, 2.0, 100.0]
{'result': 1005.0}


In [5]:
query = "calculate interest for 2000 at 10 for 1 year and then add 500"
print(execute(query))


🔍 DISCOVERED TOOLS: [{'name': 'add', 'description': 'Add two numbers', 'input_schema': {'type': 'object', 'properties': {'a': {'type': 'number'}, 'b': {'type': 'number'}}, 'required': ['a', 'b']}, 'server': 'http://localhost:8000'}, {'name': 'calculate_interest', 'description': 'Simple interest', 'input_schema': {'type': 'object', 'properties': {'principal': {'type': 'number'}, 'rate': {'type': 'number'}, 'time': {'type': 'number'}}, 'required': ['principal', 'rate', 'time']}, 'server': 'http://localhost:8001'}]
🧠 SELECTED TOOL: add
🔢 EXTRACTED NUMBERS: [2000.0, 10.0, 1.0, 500.0]
{'result': 2010.0}
